<a href="https://colab.research.google.com/github/E-tech-coder/heart-disease-prediction/blob/main/Predicting_Heart_Disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e2_path = kagglehub.competition_download('playground-series-s6e2')

print('Data source import complete.')


In [ ]:
print(playground_series_s6e2_path)

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import sklearn.metrics as metrics


In [ ]:
train = pd.read_csv("/root/.cache/kagglehub/competitions/playground-series-s6e2/train.csv")
train.info()

In [ ]:
#train.head(10)

In [ ]:
DiseaseMatch = {"Presence":1,
                "Absence":0}
train["Disease"] = train["Heart Disease"].map(DiseaseMatch)

In [ ]:
train = train.drop(columns = ["id", "Heart Disease"])

In [ ]:
train.shape

Logistic regression

In [ ]:
X = train.drop(columns = "Disease")
y = train["Disease"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns = X_train.columns, index = X_train.index)
X_test = pd.DataFrame(scaler.fit_transform(X_test), columns = X_test.columns, index = X_test.index)

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state = 42,
    solver = "liblinear",
    penalty = "l1", # Using penalty to select the most useful features
    l1_ratio = 1,
    warm_start = True, # keep the coeficients from the previous training
    C=1
)
model.fit(X_train,y_train)

In [ ]:
model.classes_ # {"Presence":1, "Absence":0}

In [ ]:
# Loop through a range of C values to decide which extent of penalty gives the best accuracy
cs = np.logspace(-8,8,30)

for c in cs :
  model = LogisticRegression(
    max_iter=1000,
    random_state = 42,
    solver = "liblinear",
    penalty = "l1", # Using penalty to select the most useful features
    warm_start = True, # keep the coeficients from the previous training
    C = c)
  model.fit(X_train,y_train)
  y_pred = model.predict(X_test)
  accuracy = accuracy_score(y_test, y_pred)
  print(f"The accuracy of C {c} is {accuracy}")

Model is underfitted with C <= 5.7361e-06, the penalty is too big.
Model is well fitted with C around 0.001, the accuracy plateau there is 0.8821.

In [ ]:
# Predict the probability of heart disease in the test dataset

test = pd.read_csv("/root/.cache/kagglehub/competitions/playground-series-s6e2/test.csv").drop(columns = "id")
submission = pd.read_csv("/root/.cache/kagglehub/competitions/playground-series-s6e2/sample_submission.csv")

In [ ]:
test = pd.DataFrame(scaler.fit_transform(test), columns = test.columns, index = test.index)

model = LogisticRegression(
    max_iter=1000,
    random_state = 42,
    solver = "liblinear",
    penalty = "l1", # Using penalty to select the most useful features
    warm_start = True, # keep the coeficients from the previous training
    C = 0.001)
model.fit(X_train,y_train)
test_prob = model.predict_proba(test)


In [ ]:
Probs = []
for i in range(len(test_prob)):
  Probs.append(round(test_prob[i][1],3))

In [ ]:
submission["Heart Disease"] = Probs

In [ ]:
submission = submission.set_index("id")

In [ ]:
submission.to_csv("Submission_heartDisease.csv")

Plot the ROC curve for the 20% test data

In [ ]:
from sklearn.metrics import roc_curve, auc
y_prob = model.predict_proba(X_test)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
print(f"AUC: {roc_auc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(fpr,tpr, label = f"AUC = {roc_auc:.3f}")
plt.plot([0,1], [0,1], linestyle = "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()